In [2]:
import os
import getpass

if "GROQ_API_KEY" not in os.environ:
  os.environ["GROQ_API_KEY"] = getpass.getpass("Entrez votre clé API Groq : ")

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

class SupportManager:
  def __init__(self):
    self.embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
    self.vector_store = None

  def ingest_pdf(self, file_path: str):
    """Charge et indexe un document PDF"""
    loader = PyPDFLoader(file_path)
    documents = loader.load()
    
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    docs = text_splitter.split_documents(documents)

    self.vector_store = Chroma.from_documents(docs, self.embeddings)
    print(f"✅ Support '{file_path}' indexé avec succès ({len(docs)} fragments).")

  def get_retriever(self):
    if self.vector_store:
      return self.vector_store.as_retriever(search_kwargs={"k": 3})
    return None

In [ ]:
class LearnerProfile:
  def __init__(self):
    self.history = []
    self.global_level = "Débutant" # Débutant, Intermédiaire, Avancé
    self.weaknesses = []

  def update_profile(self, interaction: dict):
    """
    interaction: {"exercice": "...", "score": "8/10", "commentaires": "..."}
    """
    self.history.append(interaction)

  def get_summary(self) -> str:
    if not self.history:
      return f"Nouvel apprenant. Niveau estimé : {self.global_level}."
    
    summary = f"Niveau actuel : {self.global_level}\n"
    summary += f"Historique des derniers exercices : {str(self.history[-3:])}\n"
    summary += f"Points faibles identifiés : {', '.join(self.weaknesses)}"
    return summary

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser

class ExerciseAgent:
  def __init__(self, support_manager: SupportManager, profile: LearnerProfile):
    self.llm = ChatGroq(
      model="llama-3.3-70b-versatile", 
      temperature=0.7
    )
    self.support_manager = support_manager
    self.profile = profile
    self.output_parser = StrOutputParser()

  def generate(self, config: dict) -> str:
    """
    config = {
      "mode": "evaluation" | "renforcement" | "libre",
      "format": "QCM" | "Questions ouvertes" | "Code",
      "nombre_questions": 5,
      "use_support": True | False,
      "use_history": True | False
    }
    """
    context_documents = ""
    retriever = self.support_manager.get_retriever()
    if config.get("use_support") and retriever:
      query = f"Concepts clés pour exercice {config.get('format')}"
      docs = retriever.invoke(query)
      context_documents = "\n\n".join([d.page_content for d in docs])

    learner_context = "Aucun historique disponible."
    if config.get("use_history"):
      learner_context = self.profile.get_summary()

    prompt = ChatPromptTemplate.from_messages([
      ("system", (
        "Tu es un Agent Générateur d'Exercices expert et pédagogue.\n"
        "Ton but est de créer des exercices en français, parfaitement adaptés aux consignes.\n\n"
        "CONTEXTE DES SUPPORTS FOURNIS :\n{support_context}\n\n"
        "HISTORIQUE ET NIVEAU DE L'APPRENANT :\n{learner_context}\n"
      )),
      ("user", (
        "Génère un exercice selon les paramètres suivants :\n"
        "- Mode : {mode} (si 'evaluation', propose des questions de diagnostic de niveau)\n"
        "- Format attendu : {format}\n"
        "- Nombre de questions : {nombre_questions}\n\n"
        "Fournis les questions clairement, suivies des corrections détaillées séparées à la toute fin."
      ))
    ])

    chain = prompt | self.llm | self.output_parser

    response = chain.invoke({
      "support_context": context_documents if context_documents else "Pas de support spécifique fourni. Utilise tes connaissances générales.",
      "learner_context": learner_context,
      "mode": config.get("mode", "libre"),
      "format": config.get("format", "QCM"),
      "nombre_questions": config.get("nombre_questions", 3)
    })
    
    return response

In [ ]:
support_mgr = SupportManager()
student_profile = LearnerProfile()
agent = ExerciseAgent(support_manager=support_mgr, profile=student_profile)

print("--- TEST 1 : Diagnostic de niveau initial (Groq) ---")
config_test1 = {
  "mode": "evaluation",
  "format": "QCM",
  "nombre_questions": 3,
  "use_support": False,
  "use_history": False
}
print(agent.generate(config_test1))

print("\n--- TEST 2 : Exercice ciblé sur l'historique (Groq) ---")
student_profile.global_level = "Intermédiaire"
student_profile.weaknesses = ["Algorithmique", "Complexité Temporelle (Big O)"]
student_profile.update_profile({
  "exercice": "Quiz structures de données", 
  "score": "3/10", 
  "commentaires": "Confond la complexité de la recherche dans une liste vs un dictionnaire"
})

config_test2 = {
  "mode": "renforcement",
  "format": "Questions de code ou pseudo-code",
  "nombre_questions": 2,
  "use_support": False,
  "use_history": True
}
print(agent.generate(config_test2))

/home/rayane/ai-agent/edu-mind/agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3447.86it/s]


--- TEST 1 : Diagnostic de niveau initial (Groq) ---
**Exercice d'évaluation - QCM**

**Question 1 :** Quel est le but principal d'une introduction dans un texte argumentatif ?
A) Présenter les conclusions
B) Développer les arguments
C) Attirer l'attention du lecteur et présenter le sujet
D) Résumer le texte entier

**Question 2 :** Quelle est la fonction grammaticale du mot "qui" dans la phrase suivante : "Le livre qui est sur la table est à moi" ?
A) Verbe
B) Adjectif
C) Pronom relatif
D) Adverbe

**Question 3 :** Quel est le terme qui désigne l'art de bien parler et de bien écrire en français ?
A) Rhétorique
B) Poésie
C) Grammaire
D) Littérature

**Corrections détaillées :**

1. **Réponse : C)** L'introduction d'un texte argumentatif vise à attirer l'attention du lecteur et à présenter le sujet de manière claire et concise. Elle permet de situer le contexte et de donner un aperçu de l'argumentation qui suivra.

2. **Réponse : C)** Le mot "qui" est un pronom relatif qui introduit une